In [2]:
import pandas as pd
import numpy as np
import plotly as pt
import seaborn as sns
import requests
import json


In [3]:
#!pip install pymatgen
#!pip install mp_api
#!pip install pymatgen nglview

#Initialization

In [ ]:
composition_relative_tolerance = 0.5  #0.5 #0.0001 #maximum eligible difference between element indexes in two compared chemical formulas
sheet = 'Photocatalytic dataset'
#sheet = 'Photocatalytic dataset 2'
#df = pd.read_excel("/content/drive/MyDrive/University/Artificial intelligence in chemistry/Perovskite project/Perovskite-liked-oxides-bandgap-prediction/Data/Perovskite dataset export.xlsx",sheet_name=sheet)
df = pd.read_excel("Data/Perovskite dataset export.xlsx",sheet_name=sheet)

In [42]:
df.columns

Index(['Perovskite', 'Class', 'Hill formula', 'Interlayer space composition',
       'Bandgap, eV', 'DOI', 'Materials Project ID', 'COD_ID', 'Springer_ID',
       'MP_CIF_modifier', 'COD_CIF_modifier', 'Springer_CIF_modifier',
       'Materials Project verification', 'COD verification',
       'Springer verification', 'General verification', 'MP_CIF_modified',
       'COD_CIF_modified', 'Springer_CIF_modified', 'Z', 'Z_MP', 'Z_COD',
       'Z_Springer', 'a, A', 'b, A', 'c, A', 'Symmetry group', 'd,A', 'a_MP',
       'b_MP', 'c_MP', 'a_COD', 'b_COD', 'c_COD', 'a_Springer', 'b_Springer',
       'c_Springer', 'Number of octahedrons on a layer', 'Valence electrons',
       'Volume', 'Volume_MP', 'Volume_COD', 'Volume_Springer',
       'Valence Electrons Density', 'Valence Electrons Density_MP',
       'Valence Electrons Density_COD', 'Springer_Valence Electrons Density',
       'avg s valence electrons', 'avg p valence electrons',
       'avg d valence electrons', 'avg f valence electrons'

In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 634 entries, 0 to 633
Data columns (total 89 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   Perovskite                            634 non-null    object 
 1   Class                                 0 non-null      float64
 2   Hill formula                          634 non-null    object 
 3   Interlayer space composition          0 non-null      float64
 4   Bandgap, eV                           550 non-null    float64
 5   DOI                                   616 non-null    object 
 6   Materials Project ID                  634 non-null    object 
 7   COD_ID                                634 non-null    int64  
 8   Springer_ID                           62 non-null     object 
 9   MP_CIF_modifier                       182 non-null    object 
 10  COD_CIF_modifier                      172 non-null    object 
 11  Springer_CIF_modifi

In [44]:
from pymatgen.core.structure import Structure
from pymatgen.core import Composition
from pymatgen.core.periodic_table import Element
import os
import re
import nglview as nv
from pymatgen.io.ase import AseAtomsAdaptor


In [45]:
subs_map = {
    "Ph": "C6H5",
    "Bn": "C7H7",
    #"Pr": "C3H7",
    "Bu": "C4H9",
    "Hx": "C6H13",
    "Me": "CH3",
    "Et": "C2H5",
    "Oc": "C8H17",
    "Dc": "C10H21",
}

import re

def expand_substituents(formula):
    if pd.isna(formula):
        return formula

    for abbr, full in subs_map.items():
        formula = re.sub(rf'{abbr}', full, formula)
    return formula

In [46]:
print(df.shape[0])
#df = df[~df['Perovskite'].str.contains("Nx", na=False)]
#df = df[~df['Perovskite'].str.contains("Ox", na=False)]
print(df.shape[0])
df['Perovskite'] = df['Perovskite'].apply(expand_substituents)

634
634


In [47]:
def getStructureFromCIF(cif_file_name):
  if(cif_file_name==-1):
    return 0
  file_path=f"/content/drive/MyDrive/University/Artificial intelligence in chemistry/Perovskite project/Perovskite-liked-oxides-bandgap-prediction/Data/CIF/{cif_file_name}.cif"
  file_path=f"Data/CIF/{cif_file_name}.cif"
  if os.path.exists(file_path):
    try:
      structure = Structure.from_file(file_path)
    except:
      print('ERROR: Invalid structure for ',cif_file_name)
      return None
  else:
    return None

  if(structure == None):
    return None
  return structure

In [48]:
#rewrites chemicalformula without asteriks
def stringFormulaToHillFormula(formula):
  print('String to Hill: ', formula)
  parts = formula.split('*')
  main_formula = parts[0]
  if len(parts) == 1:
    comp = Composition(main_formula)
    return comp.reduced_composition
  print("--------------------")
  print(formula)
  total_formula = Composition(main_formula)
  for i in range(1, len(parts)):
    hydrate_part = parts[i]
    print("Hydrate part: ", hydrate_part)
    match = re.match(r'([0-9]*\.?[0-9]*)?([A-Za-z0-9]+)', hydrate_part)
    if not match:
      raise ValueError(f"Cannot parse hydrate: {hydrate_part}")
    number_str = match.group(1)
    n = float(number_str) if number_str else 1.0
    molecule = match.group(2)
    print("n = ",n)
    print("molecule = ",molecule)
    comp = Composition(molecule)
    comp *= n
    total_formula = total_formula + comp
    print('Interim total formula: ',total_formula)
    print("------------")
  print('Final total formula: ',total_formula)
  return total_formula.reduced_composition

In [49]:
#print(eliminateAsterisksFromFormula("CuSO4*5H2O"))
#print(eliminateAsterisksFromFormula("CuSO4*0.25H2O"))
#print(eliminateAsterisksFromFormula("CuSO4*0.25H2O*2La2O3"))
print(stringFormulaToHillFormula("SrTiO3*0.01Cr2O3"))
print(stringFormulaToHillFormula("KCa2.37Nb3O10"))

String to Hill:  SrTiO3*0.01Cr2O3
--------------------
SrTiO3*0.01Cr2O3
Hydrate part:  0.01Cr2O3
n =  0.01
molecule =  Cr2O3
Interim total formula:  Sr1 Ti1 O3.03 Cr0.02
------------
Final total formula:  Sr1 Ti1 O3.03 Cr0.02
Sr1 Ti1 O3.03 Cr0.02
String to Hill:  KCa2.37Nb3O10
K1 Ca2.37 Nb3 O10


In [50]:
df['Hill formula'] = df['Perovskite'].apply(stringFormulaToHillFormula)

String to Hill:  SrTiO3*0.0005Al2O3
--------------------
SrTiO3*0.0005Al2O3
Hydrate part:  0.0005Al2O3
n =  0.0005
molecule =  Al2O3
Interim total formula:  Sr1 Ti1 O3.0015 Al0.001
------------
Final total formula:  Sr1 Ti1 O3.0015 Al0.001
String to Hill:  SrTiO3*0.0005Al2O3
--------------------
SrTiO3*0.0005Al2O3
Hydrate part:  0.0005Al2O3
n =  0.0005
molecule =  Al2O3
Interim total formula:  Sr1 Ti1 O3.0015 Al0.001
------------
Final total formula:  Sr1 Ti1 O3.0015 Al0.001
String to Hill:  SrTiO3*0.0005Al2O3
--------------------
SrTiO3*0.0005Al2O3
Hydrate part:  0.0005Al2O3
n =  0.0005
molecule =  Al2O3
Interim total formula:  Sr1 Ti1 O3.0015 Al0.001
------------
Final total formula:  Sr1 Ti1 O3.0015 Al0.001
String to Hill:  SrTiO3*0.0005Al2O3
--------------------
SrTiO3*0.0005Al2O3
Hydrate part:  0.0005Al2O3
n =  0.0005
molecule =  Al2O3
Interim total formula:  Sr1 Ti1 O3.0015 Al0.001
------------
Final total formula:  Sr1 Ti1 O3.0015 Al0.001
String to Hill:  SrTiO3*0.005Al2O3
-----

In [51]:


#checks whether formula corresponds to crystall composition
def checkCompositionStructureMatching(formula,cif_file_name):
  print('checkCompositionStructureMatching: entry')
  structure = getStructureFromCIF(cif_file_name)
  print('checkCompositionStructureMatching: structure is get')
  #print(structure)
  if(structure == None or structure==0):
    print('.cif file is not read')
    return False
  composition = structure.composition
  formula = stringFormulaToHillFormula(formula)
  try:
    composition_formula = Composition(formula)
  except:
    return False
  #print(type(composition))
  #print(type(composition_formula))
  #print(composition," || ", composition_formula, " = ")
  #print(composition_formula)
  #same = composition.reduced_composition == composition_formula.reduced_composition

  print(composition.items())
  factors = []
  for el, amt in composition.items():
    amt2= composition_formula[el];
    print(el," comp1: ",amt, " comp2: ",amt2)
    factor= amt2/amt
    if(factor==0):
      print("CIF file:",cif_file_name," || " ,composition," || ", composition_formula, " = ",False)
      return False
    factors.append(factor)

  factors_std = np.std(factors)
  print("Compositino scaling factors: ",factor," std:",factors_std)
  #for el, amt in comp.items():
  #      if el.symbol == from_el:
   #         for new_el, frac in to_dict.items():
  #              new_dict[Element(new_el)] = amt * frac

  #same = composition_formula.almost_equals(composition,rtol=0.4)
  same=False
  if(composition_relative_tolerance>factors_std):
    same=True

  print("CIF file:",cif_file_name," || " ,composition," || ", composition_formula, " = ",same)
  return same

In [52]:
#raise SystemExit()
#checkCompositionStructureMatching("Nb6K4O15OO","mp-560692")
#checkCompositionStructureMatching("HCa2Ta3O10*2C8H17NH2","HCa2Ta3O10_OcNH2")
checkCompositionStructureMatching("HCa2Nb3O10*2CH3NH2","HCa2Nb3O10_MeNH2")
#checkCompositionStructureMatching("K4La1.332Ta4O14","sd_1810747")

checkCompositionStructureMatching: entry
checkCompositionStructureMatching: structure is get
.cif file is not read


False

In [53]:
checkCompositionStructureMatching("La2Ti1.98Rh0.02O7","1001022")

checkCompositionStructureMatching: entry
checkCompositionStructureMatching: structure is get
String to Hill:  La2Ti1.98Rh0.02O7
dict_items([(Species La3+, 8.0), (Species Ti4+, 8.0), (Species O2-, 28.0)])
La3+  comp1:  8.0  comp2:  0
CIF file: 1001022  ||  La3+8 Ti4+8 O2-28  ||  La2 Ti1.98 Rh0.02 O7  =  False


False

# CIF file extraction from web


In [ ]:
cif_folder = "Data/CIF/"
from mp_api.client import MPRester
API_KEY=''

def extract_CIF_from_MP(material_id):
  if pd.isna(material_id) or material_id == "-1" or material_id == -1 or material_id == -2:
     print("Skipping invalid ID:", material_id)
     return
  print("ID: ", material_id)

  if not(material_id.startswith("mp-")):
     return
  #path = f'/content/drive/MyDrive/University/Artificial intelligence in chemistry/Perovskite project/Perovskite-liked-oxides-bandgap-prediction/Data/CIF/{material_id}.cif'
  file_path=cif_folder+ f"{material_id}.cif"
  if os.path.exists(file_path):
    print(f'CIF file for {material_id} already exist')
    return

  with MPRester(API_KEY) as mpr:
    #data = mpr.materials.search(material_ids=material_id)
    structure = mpr.materials.get_structure_by_material_id(material_id )
    cif_string = structure.to("struct.cif")

  ##with open(f"{material_id}.cif", "w") as f:
  #    f.write(cif_string)
  with open(file_path, 'w') as f:
      f.write(cif_string)
  print("ID: ", material_id, " done!")



In [55]:
extract_CIF_from_MP("mp-21699")

ID:  mp-21699
CIF file for mp-21699 already exist


In [56]:
def extract_cif_from_COD(COD_ID):
  if(COD_ID==-1 or COD_ID==-2 ):
    return
  print(COD_ID)
  #path = f'/content/drive/MyDrive/University/Artificial intelligence in chemistry/Perovskite project/Perovskite-liked-oxides-bandgap-prediction/Data/CIF/{COD_ID}.cif'
  path=cif_folder+ f"{COD_ID}.cif"
  if os.path.exists(path):
    print(f'CIF file for {COD_ID} already exist')
    return

  url = f"https://www.crystallography.net/cod/{COD_ID}.cif"
  response = requests.get(url)

  ##with open(f"{material_id}.cif", "w") as f:
  #    f.write(cif_string)
  if response.status_code == 200:
    print("Sucess")
    with open(path, 'w') as f:
        f.write(response.text)
  else:
    print('No CIF')

In [57]:
extract_cif_from_COD(1000022)

1000022
CIF file for 1000022 already exist


In [58]:
for material_id in df['COD_ID'].unique():
    extract_cif_from_COD(material_id)

1512124
CIF file for 1512124 already exist
1009075
CIF file for 1009075 already exist
1542205
CIF file for 1542205 already exist
9009886
CIF file for 9009886 already exist
1531431
CIF file for 1531431 already exist
1011195
CIF file for 1011195 already exist
1532743
CIF file for 1532743 already exist
1544411
CIF file for 1544411 already exist
9012322
CIF file for 9012322 already exist
1530606
CIF file for 1530606 already exist
1011144
CIF file for 1011144 already exist
1521385
CIF file for 1521385 already exist
1521093
CIF file for 1521093 already exist
1531051
CIF file for 1531051 already exist
1532747
CIF file for 1532747 already exist
1001842
CIF file for 1001842 already exist
1525846
CIF file for 1525846 already exist
1010942
CIF file for 1010942 already exist
9008090
CIF file for 9008090 already exist
4329321
CIF file for 4329321 already exist
2106523
CIF file for 2106523 already exist
1532721
CIF file for 1532721 already exist
1011064
CIF file for 1011064 already exist
1530317
CIF

In [59]:
for material_id in df['Materials Project ID'].unique():
    extract_CIF_from_MP(material_id)

ID:  mp-4651
CIF file for mp-4651 already exist
Skipping invalid ID: -1
ID:  mp-5999
CIF file for mp-5999 already exist
ID:  mp-22736
CIF file for mp-22736 already exist
ID:  mp-5238
CIF file for mp-5238 already exist
ID:  mp-3614
CIF file for mp-3614 already exist
ID:  mp-7375
CIF file for mp-7375 already exist
ID:  mp-1244890
CIF file for mp-1244890 already exist
ID:  mp-1019544
CIF file for mp-1019544 already exist
ID:  mp-1227503
CIF file for mp-1227503 already exist
ID:  mp-3491
CIF file for mp-3491 already exist
ID:  mp-1179305
CIF file for mp-1179305 already exist
ID:  mp-676280
CIF file for mp-676280 already exist
ID:  mp-12867
CIF file for mp-12867 already exist
ID:  mp-754345
CIF file for mp-754345 already exist
ID:  mp-542112
CIF file for mp-542112 already exist
ID:  mp-560692
CIF file for mp-560692 already exist
ID:  mp-674328
CIF file for mp-674328 already exist
ID:  mp-1245098
CIF file for mp-1245098 already exist
ID:  mp-1205881
CIF file for mp-1205881 already exist
ID: 

#CIF modifier


In [60]:
#converts written instructions of stoichiometric_replacement in CIF file into structures
#For example: "Ta->0.5Nb,0.5Ta"     {'from': 'Ta', 'total': 1.0, 'to': {'Nb': 0.5, 'Ta': 0.5}}
def parse_stoichiometric_replacement(expr):
  expr = expr.replace(" ", "")
  if "->" not in expr:
        raise ValueError(f"Invalid expression (missing ->): {expr}")
  lhs, rhs = expr.split("->")
  print("LHS: ",lhs," RHS: ", rhs)

  # --- Parse LHS ---
  m = re.fullmatch(r"(?:(\d+(?:\.\d+)?))?([A-Z][a-z]?)", lhs)
  if not m:
      raise ValueError(f"Invalid LHS: {lhs}")

  lhs_coeff = float(m.group(1)) if m.group(1) else 1.0
  lhs_elem = m.group(2)

  # --- Parse RHS ---
  terms = rhs.split(",")
  rhs_counts = {}
  for term in terms:
        print('Term: ', term)
        #m = re.fullmatch(r"(\d+(?:\.\d+)?)([A-Z][a-z]?)", term)
        m = re.fullmatch(r"(?:(\d+(?:\.\d+)?))?([A-Z][a-z]?)", term)
        if not m:
            raise ValueError(f"Invalid RHS term: {term}")
        print("Term goups: ", m.group(1), "  ; ", m.group(2))
        coeff = 1
        if(m.group(1) is not None):
          coeff = float(m.group(1))
        elem = m.group(2)

        rhs_counts[elem] = rhs_counts.get(elem, 0.0) + coeff
  # --- Normalize RHS ---
  total_rhs = sum(rhs_counts.values())
  if total_rhs == 0:
      raise ValueError("RHS total stoichiometry is zero")

  rhs_fractions = {
      elem: coeff / total_rhs
      for elem, coeff in rhs_counts.items()
  }

  return {
      "from": lhs_elem,
      "total": lhs_coeff,
      "to": rhs_fractions
  }

In [61]:
inp = "Ta->Nb"
com = parse_stoichiometric_replacement(inp)
print(com)

inp = "Ta->0.5Nb,0.5Ta"
com = parse_stoichiometric_replacement(inp)
print(com)

LHS:  Ta  RHS:  Nb
Term:  Nb
Term goups:  None   ;  Nb
{'from': 'Ta', 'total': 1.0, 'to': {'Nb': 1.0}}
LHS:  Ta  RHS:  0.5Nb,0.5Ta
Term:  0.5Nb
Term goups:  0.5   ;  Nb
Term:  0.5Ta
Term goups:  0.5   ;  Ta
{'from': 'Ta', 'total': 1.0, 'to': {'Nb': 0.5, 'Ta': 0.5}}


In [62]:
def replace_element(comp, from_el, to_dict):
    print("Element replacement start: From ",from_el," To: ",to_dict)
    new_dict = {}

    for el, amt in comp.items():
        if el.symbol == from_el:
            for new_el, frac in to_dict.items():
                new_dict[Element(new_el)] = amt * frac
        else:
            new_dict[el] = amt
    output = Composition(new_dict)
    print("New comp: ", output)
    print("Element replacement is done!")
    return output

def modify_structure(structure, instruction):
  print("Start structure modification!")
  if(structure is None):
    print("Null structure")
    return None
  try:
    instructions = [cmd.strip() for cmd in instruction.split(";") if cmd.strip()]
        #old, new = instruction.split("->")
        #old = old.strip()
        #new = new.strip()
  except ValueError:
    raise ValueError("Failed to separte instructinos")
        #raise ValueError("Instruction must be of the form 'A->B', e.g. 'K->H'")
  output = structure
  for command in instructions:
    parsed_command = parse_stoichiometric_replacement(command)
    print("Parsed command: ", parsed_command)
    for site in structure:
      if site.is_ordered:
        print("Ordered site:", site.specie)
        if site.specie.symbol == parsed_command["from"]:
          site.species = {
              Element(el): frac
              for el, frac in parsed_command["to"].items()  #[TO DO]: not always 1:1 replacement
          }
      else:
        print("Disordered site:", site.species)
        print(site.species)
        print(type(site.species))
        species_comp = site.species;
        new_species_comp = replace_element(species_comp, parsed_command["from"], parsed_command["to"])
        site.species = new_species_comp
  print("Finish structure modification!")
  print("-------------------------------")
  print("-------------------------------")
  print("-------------------------------")
  print("-------------------------------")
  print("-------------------------------")
  return output


In [63]:
import os
#s = getStructureFromCIF("sd_1810747")
#s
#s_new = modify_structure(s, "K->H")
s = getStructureFromCIF("sd_1958942")
s
s_new = modify_structure(s, "2Sr->Sr,Pb")
#s_new.to("new_cif.cif","cif")

Start structure modification!
LHS:  2Sr  RHS:  Sr,Pb
Term:  Sr
Term goups:  None   ;  Sr
Term:  Pb
Term goups:  None   ;  Pb
Parsed command:  {'from': 'Sr', 'total': 2.0, 'to': {'Sr': 0.5, 'Pb': 0.5}}
Disordered site: Bi0.5 Sr0.5
Bi0.5 Sr0.5
<class 'pymatgen.core.composition.Composition'>
Element replacement start: From  Sr  To:  {'Sr': 0.5, 'Pb': 0.5}
New comp:  Bi0.5 Sr0.25 Pb0.25
Element replacement is done!
Disordered site: Bi0.5 Sr0.5
Bi0.5 Sr0.5
<class 'pymatgen.core.composition.Composition'>
Element replacement start: From  Sr  To:  {'Sr': 0.5, 'Pb': 0.5}
New comp:  Bi0.5 Sr0.25 Pb0.25
Element replacement is done!
Disordered site: Bi0.5 Sr0.5
Bi0.5 Sr0.5
<class 'pymatgen.core.composition.Composition'>
Element replacement start: From  Sr  To:  {'Sr': 0.5, 'Pb': 0.5}
New comp:  Bi0.5 Sr0.25 Pb0.25
Element replacement is done!
Disordered site: Bi0.5 Sr0.5
Bi0.5 Sr0.5
<class 'pymatgen.core.composition.Composition'>
Element replacement start: From  Sr  To:  {'Sr': 0.5, 'Pb': 0.5}
New

e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:3112: UserWarning: No structure parsed for section 1 in CIF.
'_atom_site_label'
  struct = parser.parse_structures(primitive=primitive)[0]
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\io\cif.py:1057: UserWarning: No _symmetry_equiv_pos_as_xyz type key found. Spacegroup from _symmetry_space_group_name_H-M used.
  self.symmetry_operations = self.get_symops(data)  # type:ignore[assignment]
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\io\cif.py:1342: UserWarning: Cannot determine chemical composition from CIF! 'NoneType' object is not iterable
  if struct := self._get_structure(data, primitive, symmetrized, check_occu=check_occu):
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\io\cif.py:1057: UserWarning: No _symmetry_equiv_pos_as_xyz type key found. Defaulting to P1.
  self.symmetry_operations = self.get_symops

In [64]:
def modify_CIF(cif_file_name, instruction):
  print('modify CIF: entry', cif_file_name)
  structure = getStructureFromCIF(cif_file_name)
  print('modify CIF: structure is get')
  if(structure == None or structure==0):
    return None
  new_structure = modify_structure(structure, instruction)
  return new_structure

def modify_all_CIFs(cif_input_column, instruction_column, cif_output_column, prefix):
  results = []
  counter=0
  col_idx = {name: i for i, name in enumerate(df.columns)}
  cif_i = col_idx[cif_input_column]
  instr_i = col_idx[instruction_column]

  for i, row in enumerate(df.itertuples(index=False, name=None), start=1):
    cif_input = row[cif_i]
    instruction = row[instr_i]
    print("CIF input: ", cif_input, " instruction: ", instruction)
    if pd.isna(instruction):
      print("No instruction")
      results.append(cif_input)
      continue
    new_CIF = modify_CIF(cif_input,instruction)
    if(new_CIF is None):
      results.append(("Invalid structure to modify: "+str(cif_input)))
      continue
    new_CIF_name = "M_"+ prefix +str(counter)
    if sheet == 'Photocatalytic dataset 2':
      new_CIF_name = "N_"+ prefix +str(counter)
    counter = counter +1
    file_path=f"/content/drive/MyDrive/University/Artificial intelligence in chemistry/Perovskite project/Perovskite-liked-oxides-bandgap-prediction/Data/CIF/{new_CIF_name}.cif"
    file_path=f"Data/CIF/{new_CIF_name}.cif"
    new_CIF.to(file_path,"cif")
    results.append(new_CIF_name)

  df[cif_output_column] = results
  print("Modified CIFs: ", counter)


In [65]:
modify_all_CIFs("Materials Project ID", "MP_CIF_modifier", "MP_CIF_modified", "MP")

CIF input:  mp-4651  instruction:  nan
No instruction
CIF input:  mp-4651  instruction:  nan
No instruction
CIF input:  mp-4651  instruction:  nan
No instruction
CIF input:  mp-4651  instruction:  nan
No instruction
CIF input:  mp-4651  instruction:  nan
No instruction
CIF input:  mp-4651  instruction:  nan
No instruction
CIF input:  -1  instruction:  nan
No instruction
CIF input:  -1  instruction:  nan
No instruction
CIF input:  mp-4651  instruction:  nan
No instruction
CIF input:  mp-4651  instruction:  nan
No instruction
CIF input:  mp-4651  instruction:  nan
No instruction
CIF input:  mp-4651  instruction:  nan
No instruction
CIF input:  mp-4651  instruction:  nan
No instruction
CIF input:  mp-4651  instruction:  nan
No instruction
CIF input:  mp-4651  instruction:  nan
No instruction
CIF input:  mp-4651  instruction:  nan
No instruction
CIF input:  mp-4651  instruction:  nan
No instruction
CIF input:  mp-4651  instruction:  nan
No instruction
CIF input:  mp-4651  instruction:  nan

e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:2948: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['In0', 'In0', 'In1', 'In1', 'Cu2', 'Cu3', 'S4', 'S5', 'S6', 'S7']`.
  writer: Any = CifWriter(self, **kwargs)
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:2948: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['Zn0', 'Zn0', 'Zn1', 'Zn1', 'Zn2', 'Zn2', 'Zn3', 'Zn3', 'Zn4', 'Zn4', 'Zn5', 'Zn5', 'Zn6', 'Zn6', 'Zn7', 'Zn7', 'Zn8', 'Zn8', 'Zn9', 'Zn9', 'Zn10', 'Zn10', 'Zn11', 'Zn11', 'Zn12', 'Zn12', 'Zn13', 'Zn13', 'Zn14', 'Zn14', 'Zn15', 'Zn15', 'Zn16', 'Zn16', 'Zn17', 'Zn17', 'Zn18', 'Zn18', 'Zn19', 'Zn19', 'Zn20', 'Zn20', 'Zn21', 'Zn21', 'Zn22', 'Zn22', 'Zn23', 

modify CIF: structure is get
Start structure modification!
LHS:  Zn  RHS:  0.8Zn,0.2Cd
Term:  0.8Zn
Term goups:  0.8   ;  Zn
Term:  0.2Cd
Term goups:  0.2   ;  Cd
Parsed command:  {'from': 'Zn', 'total': 1.0, 'to': {'Zn': 0.8, 'Cd': 0.2}}
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site: Zn
Ordered site:

e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:2948: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['Ba0', 'Ba1', 'Zr2', 'Zr2', 'Zr3', 'Zr3', 'O4', 'O5', 'O6', 'O7', 'O8', 'O9']`.
  writer: Any = CifWriter(self, **kwargs)
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:2948: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['Sr0', 'Sr1', 'Ti2', 'Ti2', 'Ti3', 'Ti3', 'O4', 'O4', 'O5', 'O5', 'O6', 'O6', 'O7', 'O7', 'O8', 'O8', 'O9', 'O9']`.
  writer: Any = CifWriter(self, **kwargs)
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:2948: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (http

modify CIF: structure is get
Start structure modification!
LHS:  2Nb  RHS:  0.67Nb,1.33Ta
Term:  0.67Nb
Term goups:  0.67   ;  Nb
Term:  1.33Ta
Term goups:  1.33   ;  Ta
Parsed command:  {'from': 'Nb', 'total': 2.0, 'to': {'Nb': 0.335, 'Ta': 0.665}}
Ordered site: La
Ordered site: Nb
Ordered site: Nb
Ordered site: H
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Finish structure modification!
-------------------------------
-------------------------------
-------------------------------
-------------------------------
-------------------------------
CIF input:  mp-1205881  instruction:  2Nb->2Ta
modify CIF: entry mp-1205881
modify CIF: structure is get
Start structure modification!
LHS:  2Nb  RHS:  2Ta
Term:  2Ta
Term goups:  2   ;  Ta
Parsed command:  {'from': 'Nb', 'total': 2.0, 'to': {'Ta': 1.0}}
Ordered site: La
Ordered site: Nb
Ordered site: Nb
Ordered site: H
Ordered site: O
Ordered site: O
Ordered site: O
Ordered si

...

e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:2948: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['La0', 'La1', 'Fe2', 'Fe2', 'Fe3', 'Fe3', 'O4', 'O5', 'O6', 'O7', 'O8', 'O9']`.
  writer: Any = CifWriter(self, **kwargs)
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:2948: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['Sr0', 'Sr0', 'Sr1', 'Sr1', 'Ti2', 'Ti3', 'O4', 'O5', 'O6', 'O7', 'O8', 'O9']`.
  writer: Any = CifWriter(self, **kwargs)


modify CIF: structure is get
Start structure modification!
LHS:  Ta  RHS:  0.93Ta,0.07Bi
Term:  0.93Ta
Term goups:  0.93   ;  Ta
Term:  0.07Bi
Term goups:  0.07   ;  Bi
Parsed command:  {'from': 'Ta', 'total': 1.0, 'to': {'Ta': 0.93, 'Bi': 0.07}}
Ordered site: Na
Ordered site: Na
Ordered site: Na
Ordered site: Na
Ordered site: Na
Ordered site: Na
Ordered site: Na
Ordered site: Na
Ordered site: Ta
Ordered site: Ta
Ordered site: Ta
Ordered site: Ta
Ordered site: Ta
Ordered site: Ta
Ordered site: Ta
Ordered site: Ta
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Finish structure modification!
-------------------------------
-------------------------------
--

e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:2948: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['Ca0', 'Ca0', 'Ca1', 'Ca1', 'Ti2', 'Ti2', 'Ti3', 'Ti3', 'O4', 'O5', 'O6', 'O7', 'O8', 'O9']`.
  writer: Any = CifWriter(self, **kwargs)
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:2948: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['Zn0', 'Zn0', 'Zn0', 'Zn1', 'Zn1', 'Zn1', 'Zn2', 'Zn2', 'Zn2', 'Zn3', 'Zn3', 'Zn3', 'Zn4', 'Zn4', 'Zn4', 'Zn5', 'Zn5', 'Zn5', 'Zn6', 'Zn6', 'Zn6', 'Zn7', 'Zn7', 'Zn7', 'Zn8', 'Zn8', 'Zn8', 'Zn9', 'Zn9', 'Zn9', 'Zn10', 'Zn10', 'Zn10', 'Zn11', 'Zn11', 'Zn11', 'Zn12', 'Zn12', 'Zn12', 'Zn13', 'Zn13', 'Zn13', 'Zn14', 'Zn14', 'Zn14', 

modify CIF: structure is get
Start structure modification!
LHS:  6Ti  RHS:  5Ti,Zr
Term:  5Ti
Term goups:  5   ;  Ti
Term:  Zr
Term goups:  None   ;  Zr
Parsed command:  {'from': 'Ti', 'total': 6.0, 'to': {'Ti': 0.8333333333333334, 'Zr': 0.16666666666666666}}
Ordered site: Na
Ordered site: Na
Ordered site: Ti
Ordered site: Ti
Ordered site: Ti
Ordered site: Ti
Ordered site: Ti
Ordered site: Ti
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Finish structure modification!
-------------------------------
-------------------------------
-------------------------------
-------------------------------
-------------------------------
CIF input:  mp-5449  instruction:  6Ti->5Ti,Zr
modify CIF: entry mp-5449
modify CIF: structure is get
Start structure modification!
LHS:  6Ti  RHS:  5Ti,Zr
Term:  5Ti
Term goups:  5   ;  Ti
Term:  Zr
Term 

e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:2948: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['Na0', 'Na1', 'Ti2', 'Ti2', 'Ti3', 'Ti3', 'Ti4', 'Ti4', 'Ti5', 'Ti5', 'Ti6', 'Ti6', 'Ti7', 'Ti7', 'O8', 'O9', 'O10', 'O11', 'O12', 'O13', 'O14', 'O15', 'O16', 'O17', 'O18', 'O19', 'O20']`.
  writer: Any = CifWriter(self, **kwargs)
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:2948: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['La0', 'La0', 'La1', 'La1', 'Fe2', 'Fe3', 'O4', 'O5', 'O6', 'O7', 'O8', 'O9']`.
  writer: Any = CifWriter(self, **kwargs)


In [66]:
modify_all_CIFs("COD_ID", "COD_CIF_modifier", "COD_CIF_modified","COD")

CIF input:  1512124  instruction:  nan
No instruction
CIF input:  1512124  instruction:  nan
No instruction
CIF input:  1512124  instruction:  nan
No instruction
CIF input:  1512124  instruction:  nan
No instruction
CIF input:  1512124  instruction:  nan
No instruction
CIF input:  1512124  instruction:  nan
No instruction
CIF input:  -1  instruction:  nan
No instruction
CIF input:  -1  instruction:  nan
No instruction
CIF input:  1512124  instruction:  nan
No instruction
CIF input:  1512124  instruction:  nan
No instruction
CIF input:  1512124  instruction:  nan
No instruction
CIF input:  1512124  instruction:  nan
No instruction
CIF input:  1512124  instruction:  nan
No instruction
CIF input:  1512124  instruction:  nan
No instruction
CIF input:  1512124  instruction:  nan
No instruction
CIF input:  1512124  instruction:  nan
No instruction
CIF input:  1512124  instruction:  nan
No instruction
CIF input:  1512124  instruction:  nan
No instruction
CIF input:  1512124  instruction:  nan

e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:2948: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['In1', 'In1', 'In1', 'In1', 'In1', 'In1', 'In1', 'In1', 'Cu1', 'Cu1', 'Cu1', 'Cu1', 'S1', 'S1', 'S1', 'S1', 'S1', 'S1', 'S1', 'S1']`.
  writer: Any = CifWriter(self, **kwargs)
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:3112: UserWarning: Issues encountered while parsing CIF: 4 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:2948: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.h

modify CIF: structure is get
Start structure modification!
LHS:  Ta  RHS:  0.98Ta,0.02Cu
Term:  0.98Ta
Term goups:  0.98   ;  Ta
Term:  0.02Cu
Term goups:  0.02   ;  Cu
Parsed command:  {'from': 'Ta', 'total': 1.0, 'to': {'Ta': 0.98, 'Cu': 0.02}}
Ordered site: Ta
Ordered site: Ta
Ordered site: Ta
Ordered site: Ta
Ordered site: Bi
Ordered site: Bi
Ordered site: Bi
Ordered site: Bi
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Finish structure modification!
-------------------------------
-------------------------------
-------------------------------
-------------------------------
-------------------------------
CIF input:  1532721  instruction:  Ta->0.97Ta, 0.03Cu
modify CIF: entry 1532721
modify CIF: structure is get
Start structure modification!
LHS:  Ta  RHS:  0.97Ta,0.03Cu
T

e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:2948: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['K1', 'K1', 'K1', 'K1', 'K2', 'K2', 'K2', 'K2', 'K3', 'K3', 'K3', 'K3', 'K4', 'K4', 'K4', 'K4', 'Nb1', 'Nb1', 'Nb1', 'Nb1', 'Nb2', 'Nb2', 'Nb2', 'Nb2', 'Nb3', 'Nb3', 'Nb3', 'Nb3', 'Nb4', 'Nb4', 'Nb4', 'Nb4', 'Nb5', 'Nb5', 'Nb5', 'Nb5', 'Nb6', 'Nb6', 'Nb6', 'Nb6', 'O1', 'O1', 'O1', 'O1', 'O2', 'O2', 'O2', 'O2', 'O3', 'O3', 'O3', 'O3', 'O4', 'O4', 'O4', 'O4', 'O5', 'O5', 'O5', 'O5', 'O6', 'O6', 'O6', 'O6', 'O7', 'O7', 'O7', 'O7', 'O8', 'O8', 'O8', 'O8', 'O9', 'O9', 'O9', 'O9', 'O10', 'O10', 'O10', 'O10', 'O11', 'O11', 'O11', 'O11', 'O12', 'O12', 'O12', 'O12', 'O13', 'O13', 'O13', 'O13', 'O14', 'O14', 'O14', 'O14', 'O15', 'O15', 'O15', 'O15', 'O16', 'O16', 'O16', 'O16', 'O17', 'O17', 'O17', 'O17']`.
  writer: Any = CifWriter(s

CIF input:  2102087  instruction:  nan
No instruction
CIF input:  2102087  instruction:  Ta->0.9Ta, 0.1Zn
modify CIF: entry 2102087
modify CIF: structure is get
Start structure modification!
LHS:  Ta  RHS:  0.9Ta,0.1Zn
Term:  0.9Ta
Term goups:  0.9   ;  Ta
Term:  0.1Zn
Term goups:  0.1   ;  Zn
Parsed command:  {'from': 'Ta', 'total': 1.0, 'to': {'Ta': 0.9, 'Zn': 0.1}}
Ordered site: K
Ordered site: Ta
Ordered site: O
Ordered site: O
Ordered site: O
Finish structure modification!
-------------------------------
-------------------------------
-------------------------------
-------------------------------
-------------------------------
CIF input:  2102087  instruction:  Ta->0.9Ta, 0.1Y
modify CIF: entry 2102087
modify CIF: structure is get
Start structure modification!
LHS:  Ta  RHS:  0.9Ta,0.1Y
Term:  0.9Ta
Term goups:  0.9   ;  Ta
Term:  0.1Y
Term goups:  0.1   ;  Y
Parsed command:  {'from': 'Ta', 'total': 1.0, 'to': {'Ta': 0.9, 'Y': 0.1}}
Ordered site: K
Ordered site: Ta
Ordered site

...

e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:2948: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['La1', 'La1', 'La1', 'La1', 'Fe1', 'Fe1', 'Fe1', 'Fe1', 'Fe1', 'Fe1', 'Fe1', 'Fe1', 'O1', 'O1', 'O1', 'O1', 'O1', 'O1', 'O1', 'O1', 'O2', 'O2', 'O2', 'O2']`.
  writer: Any = CifWriter(self, **kwargs)
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:2948: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['sr', 'sr', 'ti', 'o', 'o', 'o']`.
  writer: Any = CifWriter(self, **kwargs)


modify CIF: structure is get
Start structure modification!
LHS:  Ta  RHS:  0.9Ta,0.1Nb
Term:  0.9Ta
Term goups:  0.9   ;  Ta
Term:  0.1Nb
Term goups:  0.1   ;  Nb
Parsed command:  {'from': 'Ta', 'total': 1.0, 'to': {'Ta': 0.9, 'Nb': 0.1}}
Ordered site: Na
Ordered site: Na
Ordered site: Na
Ordered site: Na
Ordered site: Ta
Ordered site: Ta
Ordered site: Ta
Ordered site: Ta
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Finish structure modification!
-------------------------------
-------------------------------
-------------------------------
-------------------------------
-------------------------------
CIF input:  1521385  instruction:  Ta->0.8Ta, 0.2Nb
modify CIF: entry 1521385
modify CIF: structure is get
Start structure modification!
LHS:  Ta  RHS:  0.8Ta,0.2Nb
Term:  0.8Ta
Term goups:  0.8   ;  Ta
Term:  0.2Nb
Term goups:  0.2   ;  Nb
P

e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:2948: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['Ca1', 'Ca1', 'Ca1', 'Ca1', 'Ca1', 'Ca1', 'Ca1', 'Ca1', 'Ti1', 'Ti1', 'Ti1', 'Ti1', 'Ti1', 'Ti1', 'Ti1', 'Ti1', 'O1', 'O1', 'O1', 'O1', 'O2', 'O2', 'O2', 'O2', 'O2', 'O2', 'O2', 'O2']`.
  writer: Any = CifWriter(self, **kwargs)
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:2948: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['Zn1', 'Zn1', 'Zn1', 'Zn1', 'Zn1', 'Zn1', 'S1', 'S1']`.
  writer: Any = CifWriter(self, **kwargs)
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:2948: UserWarning: Site labels are not unique, 

CIF input:  4000748  instruction:  6Ti->5Ti,Zr
modify CIF: entry 4000748
modify CIF: structure is get
Start structure modification!
LHS:  6Ti  RHS:  5Ti,Zr
Term:  5Ti
Term goups:  5   ;  Ti
Term:  Zr
Term goups:  None   ;  Zr
Parsed command:  {'from': 'Ti', 'total': 6.0, 'to': {'Ti': 0.8333333333333334, 'Zr': 0.16666666666666666}}
Ordered site: Na
Ordered site: Na
Ordered site: Na
Ordered site: Na
Ordered site: Ti
Ordered site: Ti
Ordered site: Ti
Ordered site: Ti
Ordered site: Ti
Ordered site: Ti
Ordered site: Ti
Ordered site: Ti
Ordered site: Ti
Ordered site: Ti
Ordered site: Ti
Ordered site: Ti
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered sit

e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:2948: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['La1', 'La1', 'La1', 'La1', 'La1', 'La1', 'La1', 'La1', 'Fe1', 'Fe1', 'Fe1', 'Fe1', 'O1', 'O1', 'O1', 'O1', 'O1', 'O1', 'O1', 'O1', 'O2', 'O2', 'O2', 'O2']`.
  writer: Any = CifWriter(self, **kwargs)


In [67]:
modify_all_CIFs("Springer_ID", "Springer_CIF_modifier", "Springer_CIF_modified","Springer")

CIF input:  nan  instruction:  nan
No instruction
CIF input:  nan  instruction:  nan
No instruction
CIF input:  nan  instruction:  nan
No instruction
CIF input:  nan  instruction:  nan
No instruction
CIF input:  nan  instruction:  nan
No instruction
CIF input:  nan  instruction:  nan
No instruction
CIF input:  nan  instruction:  nan
No instruction
CIF input:  nan  instruction:  nan
No instruction
CIF input:  nan  instruction:  nan
No instruction
CIF input:  nan  instruction:  nan
No instruction
CIF input:  nan  instruction:  nan
No instruction
CIF input:  nan  instruction:  nan
No instruction
CIF input:  nan  instruction:  nan
No instruction
CIF input:  nan  instruction:  nan
No instruction
CIF input:  nan  instruction:  nan
No instruction
CIF input:  nan  instruction:  nan
No instruction
CIF input:  nan  instruction:  nan
No instruction
CIF input:  nan  instruction:  nan
No instruction
CIF input:  nan  instruction:  nan
No instruction
CIF input:  nan  instruction:  nan
No instruction


e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:3112: UserWarning: Issues encountered while parsing CIF: No structure parsed for section 1 in CIF.
'_atom_site_label'
No _symmetry_equiv_pos_as_xyz type key found. Spacegroup from _symmetry_space_group_name_H-M used.
No _symmetry_equiv_pos_as_xyz type key found. Spacegroup from _symmetry_space_group_name_H-M used.
No _symmetry_equiv_pos_as_xyz type key found. Defaulting to P1.
  struct = parser.parse_structures(primitive=primitive)[0]


ERROR: Invalid structure for  sd_1610477
modify CIF: structure is get
CIF input:  sd_1610477  instruction:  2Ca->Ca, Sr
modify CIF: entry sd_1610477
ERROR: Invalid structure for  sd_1610477
modify CIF: structure is get
CIF input:  sd_1610477  instruction:  2Ca->Ca, Sr
modify CIF: entry sd_1610477
ERROR: Invalid structure for  sd_1610477
modify CIF: structure is get
CIF input:  sd_1610477  instruction:  2Ca->Ca, Sr
modify CIF: entry sd_1610477
ERROR: Invalid structure for  sd_1610477
modify CIF: structure is get
CIF input:  sd_1610477  instruction:  2Ca->Ca, Sr
modify CIF: entry sd_1610477
ERROR: Invalid structure for  sd_1610477
modify CIF: structure is get
CIF input:  sd_1610477  instruction:  2Ca->Ca, Sr
modify CIF: entry sd_1610477
ERROR: Invalid structure for  sd_1610477
modify CIF: structure is get
CIF input:  sd_1610477  instruction:  2Ca->Ca, Sr
modify CIF: entry sd_1610477
ERROR: Invalid structure for  sd_1610477
modify CIF: structure is get
CIF input:  nan  instruction:  nan
N

e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:2948: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['La2', 'La2', 'La2', 'La2', 'La2', 'La2', 'La2', 'La2', 'La2', 'La2', 'La2', 'La2', 'La2', 'La2', 'La2', 'La2', 'La1', 'La1', 'La1', 'La1', 'La1', 'La1', 'La1', 'La1', 'Ta24', 'Ta25', 'Ta26', 'Ta27', 'O2', 'O2', 'O2', 'O2', 'O2', 'O2', 'O2', 'O2', 'O2', 'O2', 'O2', 'O2', 'O2', 'O2', 'O2', 'O2', 'O3', 'O3', 'O3', 'O3', 'O3', 'O3', 'O3', 'O3', 'O1', 'O1', 'O1', 'O1']`.
  writer: Any = CifWriter(self, **kwargs)


modify CIF: structure is get
Start structure modification!
LHS:  3La  RHS:  2La,Al
Term:  2La
Term goups:  2   ;  La
Term:  Al
Term goups:  None   ;  Al
Parsed command:  {'from': 'La', 'total': 3.0, 'to': {'La': 0.6666666666666666, 'Al': 0.3333333333333333}}
Ordered site: La
Ordered site: La
Ordered site: La
Ordered site: La
Ordered site: La
Ordered site: La
Ordered site: La
Ordered site: La
Ordered site: La
Ordered site: La
Ordered site: La
Ordered site: La
Ordered site: Ta
Ordered site: Ta
Ordered site: Ta
Ordered site: Ta
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Finish structure modi

e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:2948: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['La2', 'La2', 'La2', 'La2', 'La2', 'La2', 'La2', 'La2', 'La2', 'La2', 'La2', 'La2', 'La2', 'La2', 'La2', 'La2', 'La2', 'La2', 'La2', 'La2', 'La2', 'La2', 'La2', 'La2', 'La1', 'La1', 'La1', 'La1', 'La1', 'La1', 'La1', 'La1', 'La1', 'La1', 'La1', 'La1', 'Ta', 'Ta', 'Ta', 'Ta', 'O2', 'O2', 'O2', 'O2', 'O2', 'O2', 'O2', 'O2', 'O2', 'O2', 'O2', 'O2', 'O2', 'O2', 'O2', 'O2', 'O3', 'O3', 'O3', 'O3', 'O3', 'O3', 'O3', 'O3', 'O1', 'O1', 'O1', 'O1']`.
  writer: Any = CifWriter(self, **kwargs)


modify CIF: structure is get
Start structure modification!
LHS:  4Ta  RHS:  2Ta,2Nb
Term:  2Ta
Term goups:  2   ;  Ta
Term:  2Nb
Term goups:  2   ;  Nb
Parsed command:  {'from': 'Ta', 'total': 4.0, 'to': {'Ta': 0.5, 'Nb': 0.5}}
Ordered site: Ba
Ordered site: Ba
Ordered site: Ba
Ordered site: Ba
Ordered site: Ba
Ordered site: Ta
Ordered site: Ta
Ordered site: Ta
Ordered site: Ta
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Finish structure modification!
-------------------------------
-------------------------------
-------------------------------
-------------------------------
-------------------------------
CIF input:  sd_0305968  instruction:  nan
No instruction
CIF input:  sd_0305968  instruction:  4Ta->4Nb
modify CIF: entry sd_0305968


e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:3112: UserWarning: Issues encountered while parsing CIF: 6 fractional coordinates rounded to ideal values to avoid issues with finite precision.
No structure parsed for section 1 in CIF.
'_atom_site_label'
No _symmetry_equiv_pos_as_xyz type key found. Spacegroup from _symmetry_space_group_name_H-M used.
No _symmetry_equiv_pos_as_xyz type key found. Spacegroup from _symmetry_space_group_name_H-M used.
No _symmetry_equiv_pos_as_xyz type key found. Defaulting to P1.
  struct = parser.parse_structures(primitive=primitive)[0]
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:2948: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['Ba1', 'Ba1', 'Ba2', 'Ba2', 'Ba3', 'Ta1', 'Ta1', 'Ta1', 'Ta1', 'Ta2', 'Ta2', 'Ta2', 'Ta2', 'O1', 'O1', 'O1', 'O1'

modify CIF: structure is get
Start structure modification!
LHS:  4Ta  RHS:  4Nb
Term:  4Nb
Term goups:  4   ;  Nb
Parsed command:  {'from': 'Ta', 'total': 4.0, 'to': {'Nb': 1.0}}
Ordered site: Ba
Ordered site: Ba
Ordered site: Ba
Ordered site: Ba
Ordered site: Ba
Ordered site: Ta
Ordered site: Ta
Ordered site: Ta
Ordered site: Ta
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Ordered site: O
Finish structure modification!
-------------------------------
-------------------------------
-------------------------------
-------------------------------
-------------------------------
CIF input:  nan  instruction:  nan
No instruction
CIF input:  nan  instruction:  nan
No instruction
CIF input:  nan  instruction:  nan
No instruction
CIF input:  nan  instruction:  nan
No instruction
CIF input:  nan  instruction:  nan
No

e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:2948: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['Ba', 'Ba', 'Sn2', 'O3', 'O4', 'O5']`.
  writer: Any = CifWriter(self, **kwargs)


In [68]:
df.to_excel("checkpoint_CIF_modification.xlsx")

#CIF Verification

In [69]:
#checks whether crystall structure compositions match formulas
df["Materials Project verification"] = df.apply(lambda row: checkCompositionStructureMatching(row['Perovskite'], row['MP_CIF_modified']), axis=1)

checkCompositionStructureMatching: entry
checkCompositionStructureMatching: structure is get
String to Hill:  SrTiO3*0.0005Al2O3
--------------------
SrTiO3*0.0005Al2O3
Hydrate part:  0.0005Al2O3
n =  0.0005
molecule =  Al2O3
Interim total formula:  Sr1 Ti1 O3.0015 Al0.001
------------
Final total formula:  Sr1 Ti1 O3.0015 Al0.001
dict_items([(Element Sr, 2.0), (Element Ti, 2.0), (Element O, 6.0)])
Sr  comp1:  2.0  comp2:  1.0
Ti  comp1:  2.0  comp2:  1.0
O  comp1:  6.0  comp2:  3.0015
Compositino scaling factors:  0.50025  std: 0.00011785113019774494
CIF file: mp-4651  ||  Sr2 Ti2 O6  ||  Sr1 Ti1 O3.0015 Al0.001  =  True
checkCompositionStructureMatching: entry
checkCompositionStructureMatching: structure is get
String to Hill:  SrTiO3*0.0005Al2O3
--------------------
SrTiO3*0.0005Al2O3
Hydrate part:  0.0005Al2O3
n =  0.0005
molecule =  Al2O3
Interim total formula:  Sr1 Ti1 O3.0015 Al0.001
------------
Final total formula:  Sr1 Ti1 O3.0015 Al0.001
dict_items([(Element Sr, 2.0), (Eleme

e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:3112: UserWarning: Issues encountered while parsing CIF: 16 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:3112: UserWarning: Issues encountered while parsing CIF: 20 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]


checkCompositionStructureMatching: structure is get
String to Hill:  Gd3NbO7
dict_items([(Element Gd, 6.0), (Element Nb, 2.0), (Element O, 14.0)])
Gd  comp1:  6.0  comp2:  3.0
Nb  comp1:  2.0  comp2:  1.0
O  comp1:  14.0  comp2:  7.0
Compositino scaling factors:  0.5  std: 0.0
CIF file: mp-752432  ||  Gd6 Nb2 O14  ||  Gd3 Nb1 O7  =  True
checkCompositionStructureMatching: entry
checkCompositionStructureMatching: structure is get
String to Hill:  La3NbO7
dict_items([(Element La, 12.0), (Element Nb, 4.0), (Element O, 28.0)])
La  comp1:  12.0  comp2:  3.0
Nb  comp1:  4.0  comp2:  1.0
O  comp1:  28.0  comp2:  7.0
Compositino scaling factors:  0.25  std: 0.0
CIF file: mp-560349  ||  La12 Nb4 O28  ||  La3 Nb1 O7  =  True
checkCompositionStructureMatching: entry
checkCompositionStructureMatching: structure is get
String to Hill:  La3NbO7
dict_items([(Element La, 12.0), (Element Nb, 4.0), (Element O, 28.0)])
La  comp1:  12.0  comp2:  3.0
Nb  comp1:  4.0  comp2:  1.0
O  comp1:  28.0  comp2:  7.

In [70]:
#df["COD verification"] = df.apply(lambda row: checkCompositionStructureMatching(row['Perovskite'], row['COD_ID']), axis=1)
df["COD verification"] = df.apply(lambda row: checkCompositionStructureMatching(row['Perovskite'], row['COD_CIF_modified']), axis=1)

checkCompositionStructureMatching: entry
checkCompositionStructureMatching: structure is get
String to Hill:  SrTiO3*0.0005Al2O3
--------------------
SrTiO3*0.0005Al2O3
Hydrate part:  0.0005Al2O3
n =  0.0005
molecule =  Al2O3
Interim total formula:  Sr1 Ti1 O3.0015 Al0.001
------------
Final total formula:  Sr1 Ti1 O3.0015 Al0.001
dict_items([(Element Sr, 1.0), (Element Ti, 1.0), (Element O, 3.0)])
Sr  comp1:  1.0  comp2:  1.0
Ti  comp1:  1.0  comp2:  1.0
O  comp1:  3.0  comp2:  3.0015
Compositino scaling factors:  1.0005  std: 0.00023570226039548988
CIF file: 1512124  ||  Sr1 Ti1 O3  ||  Sr1 Ti1 O3.0015 Al0.001  =  True
checkCompositionStructureMatching: entry
checkCompositionStructureMatching: structure is get
String to Hill:  SrTiO3*0.0005Al2O3
--------------------
SrTiO3*0.0005Al2O3
Hydrate part:  0.0005Al2O3
n =  0.0005
molecule =  Al2O3
Interim total formula:  Sr1 Ti1 O3.0015 Al0.001
------------
Final total formula:  Sr1 Ti1 O3.0015 Al0.001
dict_items([(Element Sr, 1.0), (Elemen

e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\io\cif.py:1342: UserWarning: Incorrect stoichiometry:
  CIF={'O': 7.0, 'Ti': 2.0, 'Y': 2.0}
  PMG={'Y': 16.024, 'Ti': 15.975999999999997, 'O': 55.99040000000002}
  ratios={'Ti': 7.987999999999999, 'Y': 8.012, 'O': 7.998628571428575}
  if struct := self._get_structure(data, primitive, symmetrized, check_occu=check_occu):
e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:3112: UserWarning: Issues encountered while parsing CIF: 6 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]


checkCompositionStructureMatching: structure is get
String to Hill:  Ba5Ta4O15
dict_items([(Element Ba, 5.0), (Element Ta, 4.0), (Element O, 15.0)])
Ba  comp1:  5.0  comp2:  5.0
Ta  comp1:  4.0  comp2:  4.0
O  comp1:  15.0  comp2:  15.0
Compositino scaling factors:  1.0  std: 0.0
CIF file: 2106435  ||  Ba5 Ta4 O15  ||  Ba5 Ta4 O15  =  True
checkCompositionStructureMatching: entry
checkCompositionStructureMatching: structure is get
String to Hill:  NaTaO3
dict_items([(Element Na, 4.0), (Element Ta, 4.0), (Element O, 12.0)])
Na  comp1:  4.0  comp2:  1.0
Ta  comp1:  4.0  comp2:  1.0
O  comp1:  12.0  comp2:  3.0
Compositino scaling factors:  0.25  std: 0.0
CIF file: 1521385  ||  Na4 Ta4 O12  ||  Na1 Ta1 O3  =  True
checkCompositionStructureMatching: entry
checkCompositionStructureMatching: structure is get
String to Hill:  NaTaO3
dict_items([(Element Na, 4.0), (Element Ta, 4.0), (Element O, 12.0)])
Na  comp1:  4.0  comp2:  1.0
Ta  comp1:  4.0  comp2:  1.0
O  comp1:  12.0  comp2:  3.0
Compo

e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:3112: UserWarning: Issues encountered while parsing CIF: 2 fractional coordinates rounded to ideal values to avoid issues with finite precision.
No _symmetry_equiv_pos_as_xyz type key found. Spacegroup from _symmetry_space_group_name_H-M used.
  struct = parser.parse_structures(primitive=primitive)[0]


checkCompositionStructureMatching: structure is get
String to Hill:  BaCo0.33Nb0.67O3
dict_items([(Element Ba, 1.0), (Element Nb, 0.67), (Element Co, 0.33), (Element O, 3.0)])
Ba  comp1:  1.0  comp2:  1.0
Nb  comp1:  0.67  comp2:  0.67
Co  comp1:  0.33  comp2:  0.33
O  comp1:  3.0  comp2:  3.0
Compositino scaling factors:  1.0  std: 0.0
CIF file: N_COD58  ||  Ba1 Nb0.67 Co0.33 O3  ||  Ba1 Co0.33 Nb0.67 O3  =  True
checkCompositionStructureMatching: entry
checkCompositionStructureMatching: structure is get
String to Hill:  SrTi0.95Cr0.05O3
dict_items([(Element Sr, 1.0), (Element Ti, 1.0), (Element O, 3.0)])
Sr  comp1:  1.0  comp2:  1.0
Ti  comp1:  1.0  comp2:  0.95
O  comp1:  3.0  comp2:  3.0
Compositino scaling factors:  1.0  std: 0.023570226039551608
CIF file: 1512124  ||  Sr1 Ti1 O3  ||  Sr1 Ti0.95 Cr0.05 O3  =  True
checkCompositionStructureMatching: entry
checkCompositionStructureMatching: structure is get
String to Hill:  SrSnO3
dict_items([(Element Sr, 4.0), (Element Sn, 4.0), (E

e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:3112: UserWarning: Issues encountered while parsing CIF: No _symmetry_equiv_pos_as_xyz type key found. Defaulting to P1.
  struct = parser.parse_structures(primitive=primitive)[0]


checkCompositionStructureMatching: structure is get
String to Hill:  Na0.65K0.35TaO3
dict_items([(Element K, 1.4), (Element Na, 2.6), (Element Ta, 4.0), (Element O, 12.0)])
K  comp1:  1.4  comp2:  0.35
Na  comp1:  2.6  comp2:  0.65
Ta  comp1:  4.0  comp2:  1.0
O  comp1:  12.0  comp2:  3.0
Compositino scaling factors:  0.25  std: 0.0
CIF file: N_COD97  ||  K1.4 Na2.6 Ta4 O12  ||  Na0.65 K0.35 Ta1 O3  =  True
checkCompositionStructureMatching: entry
checkCompositionStructureMatching: structure is get
String to Hill:  Na0.57K0.43TaO3
dict_items([(Element K, 1.72), (Element Na, 2.28), (Element Ta, 4.0), (Element O, 12.0)])
K  comp1:  1.72  comp2:  0.43
Na  comp1:  2.28  comp2:  0.57
Ta  comp1:  4.0  comp2:  1.0
O  comp1:  12.0  comp2:  3.0
Compositino scaling factors:  0.25  std: 0.0
CIF file: N_COD98  ||  K1.72 Na2.28 Ta4 O12  ||  Na0.57 K0.43 Ta1 O3  =  True
checkCompositionStructureMatching: entry
checkCompositionStructureMatching: structure is get
String to Hill:  Na0.58K0.42TaO3
dict_

e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\core\structure.py:3112: UserWarning: Issues encountered while parsing CIF: 1 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]


checkCompositionStructureMatching: structure is get
String to Hill:  GaFeO3
dict_items([(Element Ga, 6.000000000000002), (Element Fe, 6.0), (Element O, 18.0)])
Ga  comp1:  6.000000000000002  comp2:  1.0
Fe  comp1:  6.0  comp2:  1.0
O  comp1:  18.0  comp2:  3.0
Compositino scaling factors:  0.16666666666666666  std: 2.2662332591841973e-17
CIF file: 4125730  ||  Ga6 Fe6 O18  ||  Ga1 Fe1 O3  =  True
checkCompositionStructureMatching: entry
checkCompositionStructureMatching: structure is get
String to Hill:  AgSbO3
dict_items([(Species Ag+, 16.0), (Species Sb5+, 16.0), (Species O2-, 48.0)])
Ag+  comp1:  16.0  comp2:  0
CIF file: 1011279  ||  Ag+16 Sb5+16 O2-48  ||  Ag1 Sb1 O3  =  False
checkCompositionStructureMatching: entry
checkCompositionStructureMatching: structure is get
String to Hill:  NaBi0.07Ta0.93O3
dict_items([(Element Na, 4.0), (Element Ta, 3.72), (Element Bi, 0.28), (Element O, 12.0)])
Na  comp1:  4.0  comp2:  1.0
Ta  comp1:  3.72  comp2:  0.93
Bi  comp1:  0.28  comp2:  0.07


e:\Programming practice\Python\Perovskite\.venv\Lib\site-packages\pymatgen\io\cif.py:1342: UserWarning: Incorrect stoichiometry:
  CIF={'Co': 1.0, 'La': 1.0, 'O': 3.0}
  PMG={'La': 2.0, 'Co': 2.0, 'O': 9.0}
  ratios={'La': 2.0, 'O': 3.0, 'Co': 2.0}
  if struct := self._get_structure(data, primitive, symmetrized, check_occu=check_occu):


In [71]:
#df["Springer verification"] = df.apply(lambda row: checkCompositionStructureMatching(row['Perovskite'], row['Springer_ID']), axis=1)
df["Springer verification"] = df.apply(lambda row: checkCompositionStructureMatching(row['Perovskite'], row['Springer_CIF_modified']), axis=1)

checkCompositionStructureMatching: entry
checkCompositionStructureMatching: structure is get
.cif file is not read
checkCompositionStructureMatching: entry
checkCompositionStructureMatching: structure is get
.cif file is not read
checkCompositionStructureMatching: entry
checkCompositionStructureMatching: structure is get
.cif file is not read
checkCompositionStructureMatching: entry
checkCompositionStructureMatching: structure is get
.cif file is not read
checkCompositionStructureMatching: entry
checkCompositionStructureMatching: structure is get
.cif file is not read
checkCompositionStructureMatching: entry
checkCompositionStructureMatching: structure is get
.cif file is not read
checkCompositionStructureMatching: entry
checkCompositionStructureMatching: structure is get
.cif file is not read
checkCompositionStructureMatching: entry
checkCompositionStructureMatching: structure is get
.cif file is not read
checkCompositionStructureMatching: entry
checkCompositionStructureMatching: stru

In [72]:
def markEntriesWithoutVerifiedCIF(ver1, ver2, ver3):
  if(ver1 or ver2 or ver3):
    return False
  return True

In [73]:
#Entrie is considered to be verified if it has at least one eligible cif file in correspondance
df["General verification"] = df.apply(lambda row: markEntriesWithoutVerifiedCIF(row['Materials Project verification'], row['COD verification'],row['Springer verification']), axis=1)
df_filtered = df[df['General verification'] != True]
df_filtered.info()

<class 'pandas.core.frame.DataFrame'>
Index: 560 entries, 0 to 633
Data columns (total 89 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   Perovskite                            560 non-null    object 
 1   Class                                 0 non-null      float64
 2   Hill formula                          560 non-null    object 
 3   Interlayer space composition          0 non-null      float64
 4   Bandgap, eV                           482 non-null    float64
 5   DOI                                   542 non-null    object 
 6   Materials Project ID                  560 non-null    object 
 7   COD_ID                                560 non-null    int64  
 8   Springer_ID                           44 non-null     object 
 9   MP_CIF_modifier                       181 non-null    object 
 10  COD_CIF_modifier                      171 non-null    object 
 11  Springer_CIF_modifier   

In [74]:
df_filtered.shape

(560, 89)

In [75]:
df_filtered.to_excel(f"CIF_files_processing_output/dataset_CIF_processed_filtered_{sheet}.xlsx")

In [76]:
#df.to_excel("checkpoint_CIF_verification_labels.xlsx")

In [77]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 634 entries, 0 to 633
Data columns (total 89 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   Perovskite                            634 non-null    object 
 1   Class                                 0 non-null      float64
 2   Hill formula                          634 non-null    object 
 3   Interlayer space composition          0 non-null      float64
 4   Bandgap, eV                           550 non-null    float64
 5   DOI                                   616 non-null    object 
 6   Materials Project ID                  634 non-null    object 
 7   COD_ID                                634 non-null    int64  
 8   Springer_ID                           62 non-null     object 
 9   MP_CIF_modifier                       182 non-null    object 
 10  COD_CIF_modifier                      172 non-null    object 
 11  Springer_CIF_modifi